# 🍝 Il Giro de Fito - Snowflake Platform Demo

---

## 🏛️ About the Restaurant

**Il Giro de Fito** is an authentic Italian restaurant specializing in traditional cuisine from the **Toscana region**. Our establishment offers:

| 🍕 **Artisan Pizza** | 🍝 **Handmade Pasta** | 🍷 **Fine Wines** | 💧 **Alpine Spring Water** |
|:---:|:---:|:---:|:---:|
| Wood-fired perfection using centuries-old recipes | Fresh pasta made daily with imported Italian flour | Curated selection from Tuscany's finest vineyards | Crystal-clear water sourced from the Italian Alps |

### 🎯 Demo Objectives

This notebook demonstrates the full breadth of **Snowflake's platform capabilities** including:

- ✅ All native table types (Permanent, Temporary, Transient, Hybrid)
- ✅ All native view types (Standard, Secure, Materialized)
- ✅ Apache Iceberg tables (Internal)
- ✅ Stored Procedures (Java & Python)
- ✅ User Defined Functions (Python)
- ✅ Internal Stages with COPY INTO
- ✅ Streams and Tasks
- ✅ Dynamic Tables
- ✅ Role-Based Access Control (RBAC)

---

## 📋 Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [RBAC Configuration](#2-rbac-configuration)
3. [Native Tables](#3-native-tables)
4. [Native Views](#4-native-views)
5. [Apache Iceberg Tables](#5-apache-iceberg-tables)
6. [Internal Stages & COPY INTO](#6-internal-stages--copy-into)
7. [Stored Procedures](#7-stored-procedures)
8. [User Defined Functions](#8-user-defined-functions)
9. [Streams and Tasks](#9-streams-and-tasks)
10. [Dynamic Tables](#10-dynamic-tables)
11. [Demo Queries](#11-demo-queries)
12. [Cleanup](#12-cleanup)

---

# 1. Environment Setup

📦 **Creating the foundational infrastructure for Il Giro de Fito**

In [ ]:
-- ============================================================================
-- 🏗️ INFRASTRUCTURE SETUP
-- ============================================================================
-- Description: Creates the core infrastructure components for the demo
-- ============================================================================

USE ROLE ACCOUNTADMIN;

-- 📁 Create Database
CREATE OR REPLACE DATABASE GIRODEFITO
    COMMENT = 'Il Giro de Fito - Italian Restaurant Demo Database';

-- 📂 Create Schema
CREATE OR REPLACE SCHEMA GIRODEFITO.PROD
    COMMENT = 'Production schema for restaurant operations';

-- ⚙️ Create Virtual Warehouse
CREATE OR REPLACE WAREHOUSE GIRODEFITO_WH
    WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Compute warehouse for Il Giro de Fito operations';

-- Set context
USE DATABASE GIRODEFITO;
USE SCHEMA PROD;
USE WAREHOUSE GIRODEFITO_WH;

SELECT '✅ Infrastructure setup complete!' AS STATUS;

---

# 2. RBAC Configuration

🔐 **Implementing Role-Based Access Control**

| Role | Description | Access Level |
|------|-------------|-------------|
| `GIRODEFITO_ADMIN` | Full administrative access | CREATE, ALTER, DROP, SELECT, INSERT, UPDATE, DELETE |
| `GIRODEFITO_USER` | Read-only operational access | SELECT only |

In [ ]:
-- ============================================================================
-- 🔐 ROLE-BASED ACCESS CONTROL (RBAC)
-- ============================================================================

USE ROLE ACCOUNTADMIN;

-- 👑 Create ADMIN Role
CREATE OR REPLACE ROLE GIRODEFITO_ADMIN
    COMMENT = 'Administrative role with full access to Il Giro de Fito resources';

-- 👤 Create USER Role
CREATE OR REPLACE ROLE GIRODEFITO_USER
    COMMENT = 'Operational role with read-only access to Il Giro de Fito resources';

-- 🏢 WAREHOUSE PRIVILEGES
GRANT USAGE, OPERATE, MODIFY ON WAREHOUSE GIRODEFITO_WH TO ROLE GIRODEFITO_ADMIN;
GRANT USAGE ON WAREHOUSE GIRODEFITO_WH TO ROLE GIRODEFITO_USER;

-- 📁 DATABASE & SCHEMA PRIVILEGES
GRANT USAGE, CREATE SCHEMA ON DATABASE GIRODEFITO TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT USAGE ON DATABASE GIRODEFITO TO ROLE GIRODEFITO_USER;
GRANT USAGE ON SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;

-- 📊 OBJECT PRIVILEGES
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL VIEWS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL STAGES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL PROCEDURES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL FUNCTIONS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL STREAMS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL TASKS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON ALL DYNAMIC TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;

GRANT SELECT ON ALL TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;
GRANT SELECT ON ALL VIEWS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;

-- 🔮 FUTURE GRANTS
GRANT ALL PRIVILEGES ON FUTURE TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE VIEWS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE STAGES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE PROCEDURES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE FUNCTIONS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE STREAMS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE TASKS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;
GRANT ALL PRIVILEGES ON FUTURE DYNAMIC TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_ADMIN;

GRANT SELECT ON FUTURE TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;
GRANT SELECT ON FUTURE VIEWS IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;
GRANT SELECT ON FUTURE DYNAMIC TABLES IN SCHEMA GIRODEFITO.PROD TO ROLE GIRODEFITO_USER;

-- 🔗 ROLE HIERARCHY
GRANT ROLE GIRODEFITO_USER TO ROLE GIRODEFITO_ADMIN;
GRANT ROLE GIRODEFITO_ADMIN TO ROLE SYSADMIN;

SELECT '✅ RBAC configuration complete!' AS STATUS;

---

# 3. Native Tables

📊 **Demonstrating all Snowflake native table types**

| Table Type | Persistence | Time Travel | Fail-safe | Use Case |
|------------|-------------|-------------|-----------|----------|
| **Permanent** | Persistent | Yes (up to 90 days) | Yes (7 days) | Core business data |
| **Transient** | Persistent | Yes (up to 1 day) | No | Staging/ETL data |
| **Temporary** | Session only | Yes (up to 1 day) | No | Session-specific work |
| **Hybrid** | Persistent | Limited | Yes | OLTP workloads |

## 3.1 Permanent Tables

In [ ]:
-- ============================================================================
-- 🗄️ PERMANENT TABLES
-- ============================================================================

USE ROLE GIRODEFITO_ADMIN;
USE DATABASE GIRODEFITO;
USE SCHEMA PROD;
USE WAREHOUSE GIRODEFITO_WH;

-- 📍 Locations Table
CREATE OR REPLACE TABLE LOCATIONS (
    location_id         NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    location_name       VARCHAR(100) NOT NULL,
    address             VARCHAR(255),
    city                VARCHAR(100),
    region              VARCHAR(100) DEFAULT 'Toscana',
    country             VARCHAR(100) DEFAULT 'Italia',
    phone               VARCHAR(20),
    email               VARCHAR(100),
    opening_date        DATE,
    seating_capacity    NUMBER(5),
    is_active           BOOLEAN DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) DATA_RETENTION_TIME_IN_DAYS = 30
COMMENT = 'Restaurant location information';

-- 📂 Menu Categories Table
CREATE OR REPLACE TABLE MENU_CATEGORIES (
    category_id         NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    category_name       VARCHAR(50) NOT NULL,
    description         VARCHAR(500),
    display_order       NUMBER(3),
    is_active           BOOLEAN DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Menu item categories';

-- 🍝 Menu Items Table
CREATE OR REPLACE TABLE MENU_ITEMS (
    item_id             NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    category_id         NUMBER(10) NOT NULL,
    item_name           VARCHAR(100) NOT NULL,
    description         VARCHAR(1000),
    price               NUMBER(10, 2) NOT NULL,
    cost                NUMBER(10, 2),
    calories            NUMBER(6),
    is_vegetarian       BOOLEAN DEFAULT FALSE,
    is_vegan            BOOLEAN DEFAULT FALSE,
    is_gluten_free      BOOLEAN DEFAULT FALSE,
    spice_level         NUMBER(1) DEFAULT 0,
    prep_time_minutes   NUMBER(4),
    is_available        BOOLEAN DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT fk_menu_category FOREIGN KEY (category_id) REFERENCES MENU_CATEGORIES(category_id)
) COMMENT = 'Restaurant menu items';

-- 👥 Customers Table
CREATE OR REPLACE TABLE CUSTOMERS (
    customer_id         NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    first_name          VARCHAR(50) NOT NULL,
    last_name           VARCHAR(50) NOT NULL,
    email               VARCHAR(100) UNIQUE,
    phone               VARCHAR(20),
    date_of_birth       DATE,
    loyalty_points      NUMBER(10) DEFAULT 0,
    membership_tier     VARCHAR(20) DEFAULT 'Bronze',
    preferred_location  NUMBER(10),
    dietary_preferences VARCHAR(500),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Customer information and loyalty data';

-- 👨‍🍳 Staff Table
CREATE OR REPLACE TABLE STAFF (
    staff_id            NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    location_id         NUMBER(10) NOT NULL,
    first_name          VARCHAR(50) NOT NULL,
    last_name           VARCHAR(50) NOT NULL,
    role                VARCHAR(50) NOT NULL,
    email               VARCHAR(100) UNIQUE,
    phone               VARCHAR(20),
    hire_date           DATE NOT NULL,
    hourly_rate         NUMBER(8, 2),
    is_active           BOOLEAN DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT fk_staff_location FOREIGN KEY (location_id) REFERENCES LOCATIONS(location_id)
) COMMENT = 'Restaurant staff information';

-- 🧾 Orders Table
CREATE OR REPLACE TABLE ORDERS (
    order_id            NUMBER(15) AUTOINCREMENT PRIMARY KEY,
    location_id         NUMBER(10) NOT NULL,
    customer_id         NUMBER(10),
    staff_id            NUMBER(10),
    table_number        NUMBER(4),
    order_type          VARCHAR(20) NOT NULL,
    order_status        VARCHAR(20) DEFAULT 'Pending',
    order_timestamp     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    subtotal            NUMBER(12, 2),
    tax_amount          NUMBER(10, 2),
    tip_amount          NUMBER(10, 2) DEFAULT 0,
    total_amount        NUMBER(12, 2),
    payment_method      VARCHAR(30),
    notes               VARCHAR(1000),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Customer orders';

-- 📝 Order Items Table
CREATE OR REPLACE TABLE ORDER_ITEMS (
    order_item_id       NUMBER(15) AUTOINCREMENT PRIMARY KEY,
    order_id            NUMBER(15) NOT NULL,
    item_id             NUMBER(10) NOT NULL,
    quantity            NUMBER(4) NOT NULL DEFAULT 1,
    unit_price          NUMBER(10, 2) NOT NULL,
    special_instructions VARCHAR(500),
    item_status         VARCHAR(20) DEFAULT 'Ordered',
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Individual items within orders';

-- 🍷 Wine Inventory Table
CREATE OR REPLACE TABLE WINE_INVENTORY (
    wine_id             NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    wine_name           VARCHAR(200) NOT NULL,
    winery              VARCHAR(200),
    region              VARCHAR(100) DEFAULT 'Toscana',
    vintage_year        NUMBER(4),
    wine_type           VARCHAR(50),
    grape_variety       VARCHAR(200),
    alcohol_percentage  NUMBER(4, 1),
    bottle_size_ml      NUMBER(6) DEFAULT 750,
    purchase_price      NUMBER(10, 2),
    selling_price       NUMBER(10, 2),
    quantity_in_stock   NUMBER(6),
    reorder_level       NUMBER(6) DEFAULT 10,
    tasting_notes       VARCHAR(2000),
    pairing_suggestions VARCHAR(1000),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Tuscan wine inventory from regional vineyards';

-- 💧 Spring Water Products Table
CREATE OR REPLACE TABLE SPRING_WATER_PRODUCTS (
    water_id            NUMBER(10) AUTOINCREMENT PRIMARY KEY,
    product_name        VARCHAR(100) NOT NULL,
    source_location     VARCHAR(200) DEFAULT 'Italian Alps',
    bottle_size_ml      NUMBER(6),
    water_type          VARCHAR(50),
    mineral_content     VARIANT,
    ph_level            NUMBER(4, 2),
    purchase_price      NUMBER(8, 2),
    selling_price       NUMBER(8, 2),
    quantity_in_stock   NUMBER(8),
    is_available        BOOLEAN DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Alpine spring water products';

SELECT '✅ Permanent tables created!' AS STATUS;

## 3.2 Transient Tables

In [ ]:
-- ============================================================================
-- ⚡ TRANSIENT TABLES
-- ============================================================================

-- 📥 Orders Staging Table
CREATE OR REPLACE TRANSIENT TABLE ORDERS_STAGING (
    staging_id          NUMBER(15) AUTOINCREMENT,
    raw_data            VARIANT,
    source_file         VARCHAR(500),
    load_timestamp      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    is_processed        BOOLEAN DEFAULT FALSE,
    error_message       VARCHAR(2000)
) DATA_RETENTION_TIME_IN_DAYS = 1
COMMENT = 'Staging table for order data imports';

-- 📊 Daily Metrics Staging
CREATE OR REPLACE TRANSIENT TABLE DAILY_METRICS_STAGING (
    metric_date         DATE,
    location_id         NUMBER(10),
    metric_name         VARCHAR(100),
    metric_value        NUMBER(18, 4),
    load_timestamp      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) DATA_RETENTION_TIME_IN_DAYS = 1
COMMENT = 'Staging for daily operational metrics';

-- 🔄 ETL Control Log
CREATE OR REPLACE TRANSIENT TABLE ETL_CONTROL_LOG (
    log_id              NUMBER(15) AUTOINCREMENT,
    job_name            VARCHAR(200) NOT NULL,
    source_table        VARCHAR(200),
    target_table        VARCHAR(200),
    records_processed   NUMBER(15),
    records_failed      NUMBER(15) DEFAULT 0,
    start_time          TIMESTAMP_NTZ,
    end_time            TIMESTAMP_NTZ,
    status              VARCHAR(20),
    error_details       VARCHAR(4000)
) DATA_RETENTION_TIME_IN_DAYS = 1
COMMENT = 'ETL job execution log';

SELECT '✅ Transient tables created!' AS STATUS;

## 3.3 Temporary Tables

In [ ]:
-- ============================================================================
-- 🕐 TEMPORARY TABLES (Session-scoped)
-- ============================================================================

CREATE OR REPLACE TEMPORARY TABLE TEMP_ORDER_ANALYSIS (
    analysis_id         NUMBER(10) AUTOINCREMENT,
    order_id            NUMBER(15),
    calculated_margin   NUMBER(10, 2),
    performance_score   NUMBER(5, 2),
    analysis_notes      VARCHAR(1000),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Temporary table for session-based order analysis';

CREATE OR REPLACE TEMPORARY TABLE TEMP_CALCULATIONS (
    calc_id             NUMBER(10) AUTOINCREMENT,
    calc_type           VARCHAR(50),
    input_values        VARIANT,
    result_value        NUMBER(18, 6),
    calc_timestamp      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Temporary working table for complex calculations';

SELECT '✅ Temporary tables created (session-scoped)!' AS STATUS;

## 3.4 Hybrid Tables

In [ ]:
-- ============================================================================
-- ⚡ HYBRID TABLES (OLTP-optimized)
-- ============================================================================

-- 🪑 Real-time Reservations
CREATE OR REPLACE HYBRID TABLE RESERVATIONS (
    reservation_id      NUMBER(15) AUTOINCREMENT PRIMARY KEY,
    location_id         NUMBER(10) NOT NULL,
    customer_id         NUMBER(10),
    reservation_date    DATE NOT NULL,
    reservation_time    TIME NOT NULL,
    party_size          NUMBER(3) NOT NULL,
    table_number        NUMBER(4),
    status              VARCHAR(20) DEFAULT 'Confirmed',
    special_requests    VARCHAR(1000),
    contact_phone       VARCHAR(20),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    INDEX idx_date_location (reservation_date, location_id),
    INDEX idx_customer (customer_id)
) COMMENT = 'Real-time reservation management (Hybrid table for OLTP)';

-- 🍳 Kitchen Order Queue
CREATE OR REPLACE HYBRID TABLE KITCHEN_QUEUE (
    queue_id            NUMBER(15) AUTOINCREMENT PRIMARY KEY,
    order_id            NUMBER(15) NOT NULL,
    order_item_id       NUMBER(15) NOT NULL,
    location_id         NUMBER(10) NOT NULL,
    station             VARCHAR(50),
    priority            NUMBER(2) DEFAULT 5,
    status              VARCHAR(20) DEFAULT 'Queued',
    assigned_chef       NUMBER(10),
    queued_at           TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    started_at          TIMESTAMP_NTZ,
    completed_at        TIMESTAMP_NTZ,
    INDEX idx_status_location (status, location_id),
    INDEX idx_station (station)
) COMMENT = 'Real-time kitchen order queue (Hybrid table for OLTP)';

-- 💳 Active Payment Sessions
CREATE OR REPLACE HYBRID TABLE ACTIVE_PAYMENT_SESSIONS (
    session_id          VARCHAR(50) PRIMARY KEY,
    order_id            NUMBER(15) NOT NULL,
    location_id         NUMBER(10) NOT NULL,
    session_status      VARCHAR(20) DEFAULT 'Active',
    payment_method      VARCHAR(30),
    amount              NUMBER(12, 2),
    currency            VARCHAR(3) DEFAULT 'EUR',
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    expires_at          TIMESTAMP_NTZ,
    INDEX idx_order (order_id),
    INDEX idx_status (session_status)
) COMMENT = 'Active payment sessions (Hybrid table for OLTP)';

SELECT '✅ Hybrid tables created!' AS STATUS;

---

# 4. Native Views

👁️ **Demonstrating all Snowflake native view types**

## 4.1 Standard Views

In [ ]:
-- ============================================================================
-- 👁️ STANDARD VIEWS
-- ============================================================================

-- 🍝 Complete Menu View
CREATE OR REPLACE VIEW V_MENU_COMPLETE AS
SELECT 
    mi.item_id,
    mc.category_name,
    mi.item_name,
    mi.description,
    mi.price,
    mi.calories,
    CASE 
        WHEN mi.is_vegan THEN '🌱 Vegan'
        WHEN mi.is_vegetarian THEN '🥬 Vegetarian'
        ELSE '🍖 Contains Meat'
    END AS dietary_info,
    mi.is_gluten_free,
    REPEAT('🌶️', mi.spice_level) AS spice_indicator,
    mi.prep_time_minutes,
    mi.is_available
FROM MENU_ITEMS mi
JOIN MENU_CATEGORIES mc ON mi.category_id = mc.category_id
WHERE mi.is_available = TRUE AND mc.is_active = TRUE
ORDER BY mc.display_order, mi.item_name;

-- 🏢 Location Summary View
CREATE OR REPLACE VIEW V_LOCATION_SUMMARY AS
SELECT 
    l.location_id,
    l.location_name,
    l.city || ', ' || l.region AS full_location,
    l.seating_capacity,
    COUNT(DISTINCT s.staff_id) AS staff_count,
    l.opening_date,
    DATEDIFF('year', l.opening_date, CURRENT_DATE()) AS years_in_operation,
    l.is_active
FROM LOCATIONS l
LEFT JOIN STAFF s ON l.location_id = s.location_id AND s.is_active = TRUE
GROUP BY l.location_id, l.location_name, l.city, l.region, 
         l.seating_capacity, l.opening_date, l.is_active;

-- 🍷 Wine List View
CREATE OR REPLACE VIEW V_WINE_LIST AS
SELECT 
    wine_id,
    wine_name,
    winery,
    vintage_year,
    wine_type,
    grape_variety,
    alcohol_percentage || '%' AS alcohol,
    selling_price AS price_per_bottle,
    ROUND(selling_price / 5, 2) AS price_per_glass,
    quantity_in_stock,
    CASE 
        WHEN quantity_in_stock = 0 THEN '❌ Out of Stock'
        WHEN quantity_in_stock <= reorder_level THEN '⚠️ Low Stock'
        ELSE '✅ In Stock'
    END AS availability,
    tasting_notes,
    pairing_suggestions
FROM WINE_INVENTORY
ORDER BY wine_type, vintage_year DESC;

-- 👥 Customer Loyalty View
CREATE OR REPLACE VIEW V_CUSTOMER_LOYALTY AS
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS full_name,
    c.email,
    c.loyalty_points,
    c.membership_tier,
    CASE c.membership_tier
        WHEN 'Platinum' THEN '💎'
        WHEN 'Gold' THEN '🥇'
        WHEN 'Silver' THEN '🥈'
        ELSE '🥉'
    END AS tier_icon,
    COUNT(o.order_id) AS total_orders,
    COALESCE(SUM(o.total_amount), 0) AS lifetime_spend,
    MAX(o.order_timestamp) AS last_order_date
FROM CUSTOMERS c
LEFT JOIN ORDERS o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.email, 
         c.loyalty_points, c.membership_tier;

SELECT '✅ Standard views created!' AS STATUS;

## 4.2 Secure Views

In [ ]:
-- ============================================================================
-- 🔒 SECURE VIEWS
-- ============================================================================

-- 📊 Secure Revenue Report
CREATE OR REPLACE SECURE VIEW V_SECURE_REVENUE_REPORT AS
SELECT 
    l.location_name,
    DATE_TRUNC('month', o.order_timestamp) AS month,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.total_amount) AS gross_revenue,
    SUM(o.tip_amount) AS total_tips,
    AVG(o.total_amount) AS avg_order_value,
    SUM(o.subtotal) * 0.35 AS estimated_gross_profit
FROM ORDERS o
JOIN LOCATIONS l ON o.location_id = l.location_id
WHERE o.order_status = 'Paid'
GROUP BY l.location_name, DATE_TRUNC('month', o.order_timestamp);

-- 👤 Secure Customer Analytics (PII Protected)
CREATE OR REPLACE SECURE VIEW V_SECURE_CUSTOMER_ANALYTICS AS
SELECT 
    SHA2(c.customer_id::VARCHAR || c.email, 256) AS customer_hash,
    c.membership_tier,
    CASE 
        WHEN c.date_of_birth IS NOT NULL 
        THEN FLOOR(DATEDIFF('year', c.date_of_birth, CURRENT_DATE()) / 10) * 10 || 's'
        ELSE 'Unknown'
    END AS age_group,
    l.city AS preferred_city,
    COUNT(o.order_id) AS order_count,
    AVG(o.total_amount) AS avg_order_value,
    MAX(o.order_timestamp) AS last_visit
FROM CUSTOMERS c
LEFT JOIN ORDERS o ON c.customer_id = o.customer_id
LEFT JOIN LOCATIONS l ON c.preferred_location = l.location_id
GROUP BY customer_hash, c.membership_tier, c.date_of_birth, l.city;

-- 💰 Secure Pricing View
CREATE OR REPLACE SECURE VIEW V_SECURE_MENU_PRICING AS
SELECT 
    item_id,
    item_name,
    price,
    CASE 
        WHEN cost > 0 THEN ROUND((price - cost) / price * 100, 1)
        ELSE NULL
    END AS margin_pct,
    CASE 
        WHEN cost > 0 AND (price - cost) / price < 0.25 THEN '⚠️ Low Margin'
        WHEN cost > 0 AND (price - cost) / price >= 0.40 THEN '✅ High Margin'
        ELSE '➖ Standard'
    END AS margin_indicator
FROM MENU_ITEMS
WHERE is_available = TRUE;

SELECT '✅ Secure views created!' AS STATUS;

## 4.3 Materialized Views

In [ ]:
-- ============================================================================
-- 📦 MATERIALIZED VIEWS
-- ============================================================================

-- 📊 Daily Sales Summary
CREATE OR REPLACE MATERIALIZED VIEW MV_DAILY_SALES_SUMMARY AS
SELECT 
    DATE(o.order_timestamp) AS order_date,
    o.location_id,
    o.order_type,
    COUNT(DISTINCT o.order_id) AS order_count,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(o.subtotal) AS subtotal,
    SUM(o.tax_amount) AS tax_total,
    SUM(o.tip_amount) AS tips_total,
    SUM(o.total_amount) AS revenue,
    AVG(o.total_amount) AS avg_ticket_size
FROM ORDERS o
WHERE o.order_status = 'Paid'
GROUP BY DATE(o.order_timestamp), o.location_id, o.order_type;

-- 🏆 Menu Item Performance
CREATE OR REPLACE MATERIALIZED VIEW MV_MENU_ITEM_PERFORMANCE AS
SELECT 
    mi.item_id,
    mi.item_name,
    mc.category_name,
    COUNT(DISTINCT oi.order_id) AS times_ordered,
    SUM(oi.quantity) AS total_quantity_sold,
    SUM(oi.quantity * oi.unit_price) AS total_revenue,
    AVG(oi.unit_price) AS avg_selling_price
FROM ORDER_ITEMS oi
JOIN MENU_ITEMS mi ON oi.item_id = mi.item_id
JOIN MENU_CATEGORIES mc ON mi.category_id = mc.category_id
GROUP BY mi.item_id, mi.item_name, mc.category_name;

-- 💧 Inventory Status Summary
CREATE OR REPLACE MATERIALIZED VIEW MV_INVENTORY_STATUS AS
SELECT 
    'Wine' AS product_category,
    wine_type AS sub_category,
    COUNT(*) AS product_count,
    SUM(quantity_in_stock) AS total_units,
    SUM(quantity_in_stock * purchase_price) AS inventory_value,
    SUM(CASE WHEN quantity_in_stock <= reorder_level THEN 1 ELSE 0 END) AS items_below_reorder
FROM WINE_INVENTORY
GROUP BY wine_type
UNION ALL
SELECT 
    'Spring Water' AS product_category,
    water_type AS sub_category,
    COUNT(*) AS product_count,
    SUM(quantity_in_stock) AS total_units,
    SUM(quantity_in_stock * purchase_price) AS inventory_value,
    SUM(CASE WHEN quantity_in_stock < 50 THEN 1 ELSE 0 END) AS items_below_reorder
FROM SPRING_WATER_PRODUCTS
GROUP BY water_type;

SELECT '✅ Materialized views created!' AS STATUS;

---

# 5. Apache Iceberg Tables

🧊 **Open table format for interoperability**

In [ ]:
-- ============================================================================
-- 🧊 APACHE ICEBERG TABLES (Internal - Snowflake Managed)
-- ============================================================================
-- Note: External volume must be pre-configured for Iceberg tables
-- Below shows the syntax; adjust external_volume as needed
-- ============================================================================

-- 📊 Iceberg: Historical Order Archive
/*
CREATE OR REPLACE ICEBERG TABLE ICE_ORDER_ARCHIVE (
    order_id            NUMBER(15),
    location_id         NUMBER(10),
    customer_id         NUMBER(10),
    order_date          DATE,
    order_year          NUMBER(4),
    order_month         NUMBER(2),
    order_type          VARCHAR(20),
    subtotal            NUMBER(12, 2),
    tax_amount          NUMBER(10, 2),
    tip_amount          NUMBER(10, 2),
    total_amount        NUMBER(12, 2),
    payment_method      VARCHAR(30),
    archived_at         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'iceberg_external_volume'
    BASE_LOCATION = 'girodefito/order_archive/';
*/

-- Alternative: Standard table ready for Iceberg conversion
CREATE OR REPLACE TABLE ICE_ORDER_ARCHIVE_READY (
    order_id            NUMBER(15),
    location_id         NUMBER(10),
    customer_id         NUMBER(10),
    order_date          DATE,
    order_year          NUMBER(4),
    order_month         NUMBER(2),
    order_type          VARCHAR(20),
    subtotal            NUMBER(12, 2),
    tax_amount          NUMBER(10, 2),
    tip_amount          NUMBER(10, 2),
    total_amount        NUMBER(12, 2),
    payment_method      VARCHAR(30),
    archived_at         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Standard table - ready for Iceberg conversion when external volume configured';

-- 📈 Customer Behavior Analytics (Iceberg-ready)
CREATE OR REPLACE TABLE ICE_CUSTOMER_BEHAVIOR_READY (
    customer_id         NUMBER(10),
    analysis_date       DATE,
    visit_count         NUMBER(10),
    total_spend         NUMBER(14, 2),
    avg_order_value     NUMBER(12, 2),
    favorite_category   VARCHAR(50),
    favorite_item       VARCHAR(100),
    preferred_order_type VARCHAR(20),
    days_since_last_visit NUMBER(5),
    churn_risk_score    NUMBER(5, 2),
    computed_at         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Iceberg-ready table for customer behavior analytics';

-- 🍷 Wine Sales History (Iceberg-ready)
CREATE OR REPLACE TABLE ICE_WINE_SALES_HISTORY_READY (
    sale_id             NUMBER(15),
    wine_id             NUMBER(10),
    location_id         NUMBER(10),
    sale_date           DATE,
    quantity_sold       NUMBER(6),
    unit_price          NUMBER(10, 2),
    total_sale          NUMBER(12, 2),
    sale_type           VARCHAR(20),
    order_id            NUMBER(15),
    recorded_at         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Iceberg-ready table for wine sales history';

SELECT '✅ Iceberg-ready tables created!' AS STATUS;

---

# 6. Internal Stages & COPY INTO

📁 **Data loading with Snowflake internal stages**

In [ ]:
-- ============================================================================
-- 📁 INTERNAL STAGES & FILE FORMATS
-- ============================================================================

-- 📂 Create stages
CREATE OR REPLACE STAGE STG_DATA_LOAD
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Primary stage for data file uploads';

CREATE OR REPLACE STAGE STG_JSON_DATA
    FILE_FORMAT = (TYPE = JSON)
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Stage for JSON format data files';

CREATE OR REPLACE STAGE STG_CSV_DATA
    FILE_FORMAT = (
        TYPE = CSV
        FIELD_DELIMITER = ','
        SKIP_HEADER = 1
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        NULL_IF = ('NULL', 'null', '')
    )
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Stage for CSV format data files';

-- 📝 File formats
CREATE OR REPLACE FILE FORMAT FF_CSV
    TYPE = CSV
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('NULL', 'null', '')
    TRIM_SPACE = TRUE;

CREATE OR REPLACE FILE FORMAT FF_JSON
    TYPE = JSON
    STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE FILE FORMAT FF_PARQUET
    TYPE = PARQUET
    COMPRESSION = SNAPPY;

SELECT '✅ Stages and file formats created!' AS STATUS;

In [ ]:
-- ============================================================================
-- 📊 LOAD SAMPLE DATA
-- ============================================================================

-- Insert sample locations
INSERT INTO LOCATIONS (location_name, address, city, region, country, phone, email, opening_date, seating_capacity)
VALUES 
    ('Il Giro de Fito - Firenze', 'Via dei Calzaiuoli 42', 'Firenze', 'Toscana', 'Italia', '+39 055 123 4567', 'firenze@girodefito.it', '2018-03-15', 80),
    ('Il Giro de Fito - Siena', 'Piazza del Campo 15', 'Siena', 'Toscana', 'Italia', '+39 0577 234 5678', 'siena@girodefito.it', '2019-06-20', 60),
    ('Il Giro de Fito - Lucca', 'Via Fillungo 88', 'Lucca', 'Toscana', 'Italia', '+39 0583 345 6789', 'lucca@girodefito.it', '2020-09-10', 50),
    ('Il Giro de Fito - Pisa', 'Lungarno Pacinotti 27', 'Pisa', 'Toscana', 'Italia', '+39 050 456 7890', 'pisa@girodefito.it', '2021-04-01', 70);

-- Insert menu categories
INSERT INTO MENU_CATEGORIES (category_name, description, display_order)
VALUES 
    ('Antipasti', 'Traditional Tuscan starters', 1),
    ('Primi - Pasta', 'Handmade pasta dishes', 2),
    ('Primi - Risotto', 'Creamy Italian rice dishes', 3),
    ('Pizza', 'Wood-fired artisan pizzas', 4),
    ('Secondi', 'Main courses featuring local meats', 5),
    ('Contorni', 'Side dishes and vegetables', 6),
    ('Dolci', 'Traditional desserts', 7),
    ('Bevande', 'Beverages including Alpine spring water', 8);

-- Insert menu items
INSERT INTO MENU_ITEMS (category_id, item_name, description, price, cost, calories, is_vegetarian, is_vegan, is_gluten_free, spice_level, prep_time_minutes)
VALUES 
    (1, 'Bruschetta al Pomodoro', 'Grilled bread with fresh tomatoes, basil, and extra virgin olive oil', 8.50, 2.50, 180, TRUE, TRUE, FALSE, 0, 10),
    (1, 'Tagliere Toscano', 'Selection of Tuscan cured meats with pecorino cheese', 18.00, 7.00, 450, FALSE, FALSE, TRUE, 0, 5),
    (2, 'Pici al Ragù di Cinghiale', 'Hand-rolled pasta with wild boar sauce', 16.50, 5.50, 580, FALSE, FALSE, FALSE, 1, 25),
    (2, 'Pappardelle al Tartufo', 'Wide ribbon pasta with black truffle cream', 24.00, 9.00, 620, TRUE, FALSE, FALSE, 0, 20),
    (3, 'Risotto ai Funghi Porcini', 'Creamy risotto with porcini mushrooms', 18.00, 6.00, 520, TRUE, FALSE, TRUE, 0, 25),
    (4, 'Margherita DOP', 'San Marzano tomatoes, mozzarella di bufala, fresh basil', 14.00, 4.00, 750, TRUE, FALSE, FALSE, 0, 15),
    (4, 'Diavola', 'Spicy salami, mozzarella, chili oil', 15.50, 5.00, 850, FALSE, FALSE, FALSE, 3, 15),
    (5, 'Bistecca alla Fiorentina', 'T-bone steak Florentine style (1kg)', 55.00, 25.00, 1200, FALSE, FALSE, TRUE, 0, 30),
    (6, 'Fagioli all''Olio', 'White beans with olive oil and sage', 7.00, 2.00, 180, TRUE, TRUE, TRUE, 0, 5),
    (7, 'Tiramisù della Casa', 'House-made tiramisu', 9.00, 2.50, 380, TRUE, FALSE, FALSE, 0, 5),
    (8, 'Acqua delle Alpi - Naturale', 'Still Alpine spring water (750ml)', 5.00, 1.00, 0, TRUE, TRUE, TRUE, 0, 1),
    (8, 'Acqua delle Alpi - Frizzante', 'Sparkling Alpine spring water (750ml)', 5.50, 1.20, 0, TRUE, TRUE, TRUE, 0, 1);

SELECT '✅ Sample data loaded!' AS STATUS;

In [ ]:
-- ============================================================================
-- 🍷 WINE & WATER INVENTORY DATA
-- ============================================================================

INSERT INTO WINE_INVENTORY (wine_name, winery, region, vintage_year, wine_type, grape_variety, alcohol_percentage, purchase_price, selling_price, quantity_in_stock, tasting_notes, pairing_suggestions)
VALUES 
    ('Brunello di Montalcino Riserva', 'Biondi-Santi', 'Toscana', 2016, 'Red', 'Sangiovese', 14.5, 85.00, 180.00, 24, 'Complex aromas of dark cherry, leather, and tobacco', 'Bistecca alla Fiorentina, aged cheeses'),
    ('Chianti Classico Gran Selezione', 'Castello di Ama', 'Toscana', 2018, 'Red', 'Sangiovese', 14.0, 45.00, 95.00, 48, 'Vibrant red fruit with hints of violet and spice', 'Pici al Ragù, roasted meats'),
    ('Vernaccia di San Gimignano', 'Teruzzi & Puthod', 'Toscana', 2022, 'White', 'Vernaccia', 12.5, 14.00, 32.00, 60, 'Crisp and refreshing with citrus and almond notes', 'Seafood, light appetizers'),
    ('Rosso di Montalcino', 'Col d''Orcia', 'Toscana', 2020, 'Red', 'Sangiovese', 13.5, 22.00, 48.00, 42, 'Fresh and fruity, approachable young', 'Pizza, pasta with tomato sauce'),
    ('Vin Santo del Chianti', 'Fèlsina', 'Toscana', 2015, 'Dessert', 'Trebbiano, Malvasia', 16.0, 38.00, 85.00, 18, 'Amber color with dried fruit, honey, and hazelnut', 'Cantucci, desserts'),
    ('Prosecco Superiore DOCG', 'Nino Franco', 'Veneto', 2022, 'Sparkling', 'Glera', 11.5, 18.00, 42.00, 72, 'Fine bubbles with green apple and white flowers', 'Aperitivo, light appetizers'),
    ('Rosé Toscana IGT', 'Castello di Monsanto', 'Toscana', 2022, 'Rosé', 'Sangiovese', 12.5, 16.00, 38.00, 30, 'Pale salmon color with strawberry and citrus', 'Bruschetta, seafood salads');

INSERT INTO SPRING_WATER_PRODUCTS (product_name, source_location, bottle_size_ml, water_type, mineral_content, ph_level, purchase_price, selling_price, quantity_in_stock)
VALUES 
    ('Fonte Alpina Natural', 'Dolomiti, Italian Alps', 750, 'Still', '{"calcium": 80, "magnesium": 26, "sodium": 6}', 7.4, 1.20, 5.00, 200),
    ('Fonte Alpina Sparkling', 'Dolomiti, Italian Alps', 750, 'Sparkling', '{"calcium": 80, "magnesium": 26, "sodium": 6}', 7.2, 1.40, 5.50, 180),
    ('Fonte Alpina Natural', 'Dolomiti, Italian Alps', 500, 'Still', '{"calcium": 80, "magnesium": 26, "sodium": 6}', 7.4, 0.80, 3.50, 300);

SELECT '✅ Wine and water inventory loaded!' AS STATUS;

In [ ]:
-- ============================================================================
-- 👥 STAFF & CUSTOMER DATA
-- ============================================================================

INSERT INTO STAFF (location_id, first_name, last_name, role, email, phone, hire_date, hourly_rate)
VALUES 
    (1, 'Marco', 'Bianchi', 'Head Chef', 'marco.bianchi@girodefito.it', '+39 333 111 2222', '2018-03-15', 28.00),
    (1, 'Lucia', 'Rossi', 'Sous Chef', 'lucia.rossi@girodefito.it', '+39 333 111 3333', '2018-04-01', 22.00),
    (1, 'Giovanni', 'Ferrari', 'Manager', 'giovanni.ferrari@girodefito.it', '+39 333 111 4444', '2018-03-15', 25.00),
    (2, 'Francesco', 'Moretti', 'Head Chef', 'francesco.moretti@girodefito.it', '+39 333 222 1111', '2019-06-20', 26.00),
    (2, 'Chiara', 'Rizzo', 'Manager', 'chiara.rizzo@girodefito.it', '+39 333 222 2222', '2019-06-20', 24.00),
    (3, 'Alessandro', 'Bruno', 'Head Chef', 'alessandro.bruno@girodefito.it', '+39 333 333 1111', '2020-09-10', 25.00),
    (4, 'Roberto', 'Costa', 'Head Chef', 'roberto.costa@girodefito.it', '+39 333 444 1111', '2021-04-01', 25.00);

INSERT INTO CUSTOMERS (first_name, last_name, email, phone, date_of_birth, loyalty_points, membership_tier, preferred_location, dietary_preferences)
VALUES 
    ('Sofia', 'Lombardi', 'sofia.lombardi@email.it', '+39 340 111 1111', '1985-03-15', 2500, 'Gold', 1, 'No shellfish'),
    ('Andrea', 'Esposito', 'andrea.esposito@email.it', '+39 340 222 2222', '1990-07-22', 5200, 'Platinum', 1, NULL),
    ('Francesca', 'De Luca', 'francesca.deluca@email.it', '+39 340 333 3333', '1978-11-08', 1200, 'Silver', 2, 'Vegetarian'),
    ('Lorenzo', 'Barbieri', 'lorenzo.barbieri@email.it', '+39 340 444 4444', '1995-02-28', 800, 'Bronze', 2, NULL),
    ('Martina', 'Pellegrini', 'martina.pellegrini@email.it', '+39 340 555 5555', '1988-09-14', 3100, 'Gold', 3, 'Gluten-free');

SELECT '✅ Staff and customer data loaded!' AS STATUS;

In [ ]:
-- ============================================================================
-- 📥 COPY INTO DEMONSTRATION
-- ============================================================================

-- Generate sample orders and export to stage
CREATE OR REPLACE TEMPORARY TABLE TEMP_ORDERS_EXPORT AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY RANDOM()) AS order_id,
    UNIFORM(1, 4, RANDOM()) AS location_id,
    UNIFORM(1, 5, RANDOM()) AS customer_id,
    UNIFORM(1, 7, RANDOM()) AS staff_id,
    UNIFORM(1, 20, RANDOM()) AS table_number,
    CASE UNIFORM(1, 3, RANDOM()) WHEN 1 THEN 'Dine-in' WHEN 2 THEN 'Takeout' ELSE 'Delivery' END AS order_type,
    'Paid' AS order_status,
    DATEADD('day', -UNIFORM(0, 90, RANDOM()), CURRENT_TIMESTAMP()) AS order_timestamp,
    ROUND(UNIFORM(25, 150, RANDOM()) + RANDOM() * 50, 2) AS subtotal,
    CASE UNIFORM(1, 4, RANDOM()) WHEN 1 THEN 'Cash' WHEN 2 THEN 'Credit Card' WHEN 3 THEN 'Debit Card' ELSE 'Mobile Pay' END AS payment_method
FROM TABLE(GENERATOR(ROWCOUNT => 200));

-- Export to stage
COPY INTO @STG_CSV_DATA/orders_batch_
FROM (
    SELECT order_id, location_id, customer_id, staff_id, table_number,
           order_type, order_status, order_timestamp, subtotal,
           ROUND(subtotal * 0.10, 2) AS tax_amount,
           ROUND(subtotal * 0.15, 2) AS tip_amount,
           ROUND(subtotal * 1.25, 2) AS total_amount,
           payment_method
    FROM TEMP_ORDERS_EXPORT
)
FILE_FORMAT = (TYPE = CSV, COMPRESSION = NONE)
HEADER = TRUE
OVERWRITE = TRUE;

-- List files in stage
LIST @STG_CSV_DATA;

In [ ]:
-- ============================================================================
-- 📥 COPY INTO - Load from stage
-- ============================================================================

CREATE OR REPLACE TABLE ORDERS_FROM_STAGE (
    order_id            NUMBER(15),
    location_id         NUMBER(10),
    customer_id         NUMBER(10),
    staff_id            NUMBER(10),
    table_number        NUMBER(4),
    order_type          VARCHAR(20),
    order_status        VARCHAR(20),
    order_timestamp     TIMESTAMP_NTZ,
    subtotal            NUMBER(12, 2),
    tax_amount          NUMBER(10, 2),
    tip_amount          NUMBER(10, 2),
    total_amount        NUMBER(12, 2),
    payment_method      VARCHAR(30),
    load_timestamp      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Orders loaded via COPY INTO from internal stage';

COPY INTO ORDERS_FROM_STAGE (
    order_id, location_id, customer_id, staff_id, table_number,
    order_type, order_status, order_timestamp, subtotal, tax_amount,
    tip_amount, total_amount, payment_method
)
FROM @STG_CSV_DATA/
FILE_FORMAT = FF_CSV
PATTERN = '.*orders_batch.*\.csv.*'
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

-- Verify
SELECT COUNT(*) AS records_loaded, SUM(total_amount) AS total_revenue
FROM ORDERS_FROM_STAGE;

SELECT '✅ COPY INTO demonstration complete!' AS STATUS;

---

# 7. Stored Procedures

⚙️ **Server-side logic with Java and Python**

In [ ]:
-- ============================================================================
-- ☕ JAVA STORED PROCEDURE
-- ============================================================================

CREATE OR REPLACE PROCEDURE SP_CALCULATE_ORDER_STATS_JAVA(
    P_LOCATION_ID NUMBER,
    P_START_DATE DATE,
    P_END_DATE DATE
)
RETURNS VARIANT
LANGUAGE JAVA
RUNTIME_VERSION = '11'
PACKAGES = ('com.snowflake:snowpark:latest')
HANDLER = 'OrderStatsHandler.calculate'
AS
$$
import com.snowflake.snowpark_java.*;
import java.util.HashMap;
import java.util.Map;

public class OrderStatsHandler {
    public static Map<String, Object> calculate(Session session, 
                                                  long locationId, 
                                                  String startDate, 
                                                  String endDate) {
        String query = String.format(
            "SELECT COUNT(*) AS total_orders, SUM(total_amount) AS total_revenue, " +
            "AVG(total_amount) AS avg_order_value " +
            "FROM GIRODEFITO.PROD.ORDERS_FROM_STAGE " +
            "WHERE location_id = %d AND DATE(order_timestamp) BETWEEN '%s' AND '%s'",
            locationId, startDate, endDate
        );
        
        Row[] results = session.sql(query).collect();
        Map<String, Object> stats = new HashMap<>();
        
        if (results.length > 0) {
            Row row = results[0];
            stats.put("location_id", locationId);
            stats.put("period", startDate + " to " + endDate);
            stats.put("total_orders", row.getLong(0));
            stats.put("total_revenue", row.getDouble(1));
            stats.put("avg_order_value", row.getDouble(2));
            stats.put("status", "SUCCESS");
        }
        return stats;
    }
}
$$;

SELECT '✅ Java stored procedure created!' AS STATUS;

In [ ]:
-- ============================================================================
-- 🐍 PYTHON STORED PROCEDURES
-- ============================================================================

-- Daily Report Generator
CREATE OR REPLACE PROCEDURE SP_GENERATE_DAILY_REPORT_PYTHON(P_REPORT_DATE DATE)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'generate_report'
AS
$$
from snowflake.snowpark import Session
from datetime import datetime

def generate_report(session: Session, report_date: str) -> dict:
    query = f"""
        SELECT l.location_name, COUNT(DISTINCT o.order_id) as order_count,
               SUM(o.total_amount) as revenue, AVG(o.total_amount) as avg_ticket
        FROM GIRODEFITO.PROD.ORDERS_FROM_STAGE o
        JOIN GIRODEFITO.PROD.LOCATIONS l ON o.location_id = l.location_id
        WHERE DATE(o.order_timestamp) = '{report_date}'
        GROUP BY l.location_name
    """
    df = session.sql(query).to_pandas()
    
    report = {
        "report_date": report_date,
        "generated_at": datetime.now().isoformat(),
        "summary": {
            "total_orders": int(df['ORDER_COUNT'].sum()) if not df.empty else 0,
            "total_revenue": float(df['REVENUE'].sum()) if not df.empty else 0.0
        },
        "locations": []
    }
    
    for _, row in df.iterrows():
        report["locations"].append({
            "name": row['LOCATION_NAME'],
            "orders": int(row['ORDER_COUNT']),
            "revenue": float(row['REVENUE']),
            "avg_ticket": round(float(row['AVG_TICKET']), 2)
        })
    return report
$$;

-- Wine Reorder Alert
CREATE OR REPLACE PROCEDURE SP_WINE_REORDER_ALERT_PYTHON()
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'check_reorder'
AS
$$
from snowflake.snowpark import Session
from datetime import datetime

def check_reorder(session: Session) -> dict:
    query = """
        SELECT wine_id, wine_name, winery, wine_type, quantity_in_stock, 
               reorder_level, purchase_price
        FROM GIRODEFITO.PROD.WINE_INVENTORY
        WHERE quantity_in_stock <= reorder_level
        ORDER BY quantity_in_stock ASC
    """
    df = session.sql(query).to_pandas()
    
    alerts = {
        "generated_at": datetime.now().isoformat(),
        "alert_count": len(df),
        "wines_to_reorder": []
    }
    
    for _, row in df.iterrows():
        urgency = "CRITICAL" if row['QUANTITY_IN_STOCK'] == 0 else "HIGH" if row['QUANTITY_IN_STOCK'] <= row['REORDER_LEVEL']/2 else "MEDIUM"
        alerts["wines_to_reorder"].append({
            "wine_id": int(row['WINE_ID']),
            "wine_name": row['WINE_NAME'],
            "current_stock": int(row['QUANTITY_IN_STOCK']),
            "reorder_level": int(row['REORDER_LEVEL']),
            "urgency": urgency
        })
    return alerts
$$;

SELECT '✅ Python stored procedures created!' AS STATUS;

In [ ]:
-- Test stored procedures
CALL SP_CALCULATE_ORDER_STATS_JAVA(1, '2024-01-01', '2024-12-31');
CALL SP_GENERATE_DAILY_REPORT_PYTHON(CURRENT_DATE() - 7);
CALL SP_WINE_REORDER_ALERT_PYTHON();

---

# 8. User Defined Functions

🔧 **Custom functions in Python**

In [ ]:
-- ============================================================================
-- 🐍 PYTHON USER DEFINED FUNCTIONS
-- ============================================================================

-- 💰 Calculate Loyalty Points
CREATE OR REPLACE FUNCTION UDF_CALCULATE_LOYALTY_POINTS(order_amount FLOAT, membership_tier VARCHAR)
RETURNS NUMBER
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'calculate_points'
AS
$$
def calculate_points(order_amount: float, membership_tier: str) -> int:
    if order_amount is None or order_amount <= 0:
        return 0
    multipliers = {'Bronze': 1.0, 'Silver': 1.25, 'Gold': 1.5, 'Platinum': 2.0}
    return int(order_amount * multipliers.get(membership_tier, 1.0))
$$;

-- 🍷 Wine Pairing Recommendation
CREATE OR REPLACE FUNCTION UDF_WINE_PAIRING(dish_category VARCHAR, is_spicy BOOLEAN, budget_level VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'recommend_wine'
AS
$$
def recommend_wine(dish_category: str, is_spicy: bool, budget_level: str) -> str:
    if is_spicy:
        return 'Rosé Toscana IGT - Perfect for spicy dishes'
    
    pairings = {
        'Antipasti': {'default': 'Vernaccia di San Gimignano', 'premium': 'Prosecco Superiore DOCG'},
        'Primi - Pasta': {'default': 'Chianti Classico', 'premium': 'Brunello di Montalcino'},
        'Pizza': {'default': 'Rosso di Montalcino', 'premium': 'Chianti Classico'},
        'Secondi': {'default': 'Chianti Classico', 'premium': 'Brunello di Montalcino'},
        'Dolci': {'default': 'Vin Santo del Chianti', 'premium': 'Vin Santo del Chianti'}
    }
    cat = pairings.get(dish_category, {'default': 'Chianti Classico', 'premium': 'Brunello di Montalcino'})
    return cat['premium'] if budget_level == 'Premium' else cat['default']
$$;

-- 📊 Format Italian Currency
CREATE OR REPLACE FUNCTION UDF_FORMAT_CURRENCY_IT(amount FLOAT)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'format_currency'
AS
$$
def format_currency(amount: float) -> str:
    if amount is None:
        return '€0,00'
    integer_part = int(amount)
    decimal_part = int((amount - integer_part) * 100)
    formatted_int = f"{integer_part:,}".replace(',', '.')
    return f"€{formatted_int},{decimal_part:02d}"
$$;

-- 🌡️ Seasonal Menu Flag
CREATE OR REPLACE FUNCTION UDF_SEASONAL_MENU_FLAG(item_category VARCHAR, current_month NUMBER)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'get_seasonal_flag'
AS
$$
def get_seasonal_flag(category: str, month: int) -> str:
    winter, summer = [12, 1, 2], [6, 7, 8]
    if month in winter and category in ['Primi - Pasta', 'Secondi']:
        return '❄️ Perfect for Winter'
    elif month in summer and category in ['Antipasti', 'Bevande']:
        return '☀️ Summer Favorite'
    return ''
$$;

SELECT '✅ Python UDFs created!' AS STATUS;

In [ ]:
-- Test UDFs
SELECT 
    UDF_CALCULATE_LOYALTY_POINTS(100.00, 'Gold') AS points_earned,
    UDF_WINE_PAIRING('Secondi', FALSE, 'Premium') AS wine_pairing,
    UDF_FORMAT_CURRENCY_IT(1234.56) AS formatted_price,
    UDF_SEASONAL_MENU_FLAG('Secondi', 1) AS seasonal_flag;

---

# 9. Streams and Tasks

🔄 **Change data capture and automated processing**

In [ ]:
-- ============================================================================
-- 🔄 STREAMS
-- ============================================================================

CREATE OR REPLACE STREAM STREAM_ORDERS_CDC
    ON TABLE ORDERS_FROM_STAGE
    APPEND_ONLY = FALSE
    COMMENT = 'CDC stream for order changes';

CREATE OR REPLACE STREAM STREAM_CUSTOMERS_CDC
    ON TABLE CUSTOMERS
    APPEND_ONLY = FALSE
    COMMENT = 'CDC stream for customer data changes';

CREATE OR REPLACE STREAM STREAM_WINE_INVENTORY_CDC
    ON TABLE WINE_INVENTORY
    APPEND_ONLY = FALSE
    COMMENT = 'CDC stream for wine inventory changes';

-- CDC History Tables
CREATE OR REPLACE TABLE ORDER_CHANGE_HISTORY (
    change_id           NUMBER(15) AUTOINCREMENT,
    order_id            NUMBER(15),
    change_type         VARCHAR(10),
    new_status          VARCHAR(20),
    new_total           NUMBER(12, 2),
    changed_at          TIMESTAMP_NTZ,
    processed_at        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Historical record of order changes';

CREATE OR REPLACE TABLE INVENTORY_ALERTS (
    alert_id            NUMBER(15) AUTOINCREMENT,
    product_type        VARCHAR(50),
    product_id          NUMBER(10),
    product_name        VARCHAR(200),
    alert_type          VARCHAR(50),
    current_quantity    NUMBER(8),
    threshold_quantity  NUMBER(8),
    alert_timestamp     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    is_resolved         BOOLEAN DEFAULT FALSE
) COMMENT = 'Inventory alert notifications';

SELECT '✅ Streams created!' AS STATUS;

In [ ]:
-- ============================================================================
-- ⏰ TASKS
-- ============================================================================

-- Process Order Changes
CREATE OR REPLACE TASK TASK_PROCESS_ORDER_CHANGES
    WAREHOUSE = GIRODEFITO_WH
    SCHEDULE = '5 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('STREAM_ORDERS_CDC')
AS
INSERT INTO ORDER_CHANGE_HISTORY (order_id, change_type, new_status, new_total, changed_at)
SELECT 
    order_id,
    CASE WHEN METADATA$ACTION = 'INSERT' AND METADATA$ISUPDATE = FALSE THEN 'INSERT'
         WHEN METADATA$ACTION = 'INSERT' AND METADATA$ISUPDATE = TRUE THEN 'UPDATE'
         ELSE 'DELETE' END,
    order_status,
    total_amount,
    CURRENT_TIMESTAMP()
FROM STREAM_ORDERS_CDC;

-- Check Wine Inventory
CREATE OR REPLACE TASK TASK_CHECK_WINE_INVENTORY
    WAREHOUSE = GIRODEFITO_WH
    SCHEDULE = '30 MINUTE'
AS
INSERT INTO INVENTORY_ALERTS (product_type, product_id, product_name, alert_type, current_quantity, threshold_quantity)
SELECT 'Wine', wine_id, wine_name,
       CASE WHEN quantity_in_stock = 0 THEN 'OUT_OF_STOCK'
            WHEN quantity_in_stock <= reorder_level/2 THEN 'CRITICAL_LOW'
            ELSE 'REORDER_NEEDED' END,
       quantity_in_stock, reorder_level
FROM WINE_INVENTORY
WHERE quantity_in_stock <= reorder_level
  AND wine_id NOT IN (SELECT product_id FROM INVENTORY_ALERTS WHERE product_type = 'Wine' AND is_resolved = FALSE);

-- Enable tasks
ALTER TASK TASK_CHECK_WINE_INVENTORY RESUME;
ALTER TASK TASK_PROCESS_ORDER_CHANGES RESUME;

SELECT '✅ Tasks created and resumed!' AS STATUS;

-- View tasks
SHOW TASKS IN SCHEMA GIRODEFITO.PROD;

---

# 10. Dynamic Tables

🔄 **Declarative data pipelines with automatic refresh**

In [ ]:
-- ============================================================================
-- 🔄 DYNAMIC TABLES
-- ============================================================================

-- Sales Dashboard
CREATE OR REPLACE DYNAMIC TABLE DT_SALES_DASHBOARD
    TARGET_LAG = '5 minutes'
    WAREHOUSE = GIRODEFITO_WH
AS
SELECT 
    l.location_id, l.location_name, l.city,
    DATE(o.order_timestamp) AS order_date,
    HOUR(o.order_timestamp) AS order_hour,
    o.order_type,
    COUNT(DISTINCT o.order_id) AS order_count,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(o.total_amount) AS revenue,
    AVG(o.total_amount) AS avg_ticket_size
FROM ORDERS_FROM_STAGE o
JOIN LOCATIONS l ON o.location_id = l.location_id
GROUP BY l.location_id, l.location_name, l.city,
         DATE(o.order_timestamp), HOUR(o.order_timestamp), o.order_type;

-- Menu Rankings
CREATE OR REPLACE DYNAMIC TABLE DT_MENU_RANKINGS
    TARGET_LAG = '10 minutes'
    WAREHOUSE = GIRODEFITO_WH
AS
SELECT 
    mi.item_id, mi.item_name, mc.category_name, mi.price,
    COUNT(DISTINCT oi.order_id) AS times_ordered,
    SUM(oi.quantity) AS total_quantity,
    SUM(oi.quantity * oi.unit_price) AS total_revenue,
    RANK() OVER (PARTITION BY mc.category_name ORDER BY SUM(oi.quantity) DESC) AS category_rank,
    RANK() OVER (ORDER BY SUM(oi.quantity) DESC) AS overall_rank
FROM MENU_ITEMS mi
JOIN MENU_CATEGORIES mc ON mi.category_id = mc.category_id
LEFT JOIN ORDER_ITEMS oi ON mi.item_id = oi.item_id
GROUP BY mi.item_id, mi.item_name, mc.category_name, mi.price;

-- Customer LTV
CREATE OR REPLACE DYNAMIC TABLE DT_CUSTOMER_LTV
    TARGET_LAG = '30 minutes'
    WAREHOUSE = GIRODEFITO_WH
AS
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    c.membership_tier, c.loyalty_points,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.total_amount) AS lifetime_revenue,
    AVG(o.total_amount) AS avg_order_value,
    (COUNT(o.order_id) * 10) + (COALESCE(SUM(o.total_amount), 0) / 10) + (c.loyalty_points / 100) AS ltv_score
FROM CUSTOMERS c
LEFT JOIN ORDERS_FROM_STAGE o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.membership_tier, c.loyalty_points;

-- Wine Inventory Status
CREATE OR REPLACE DYNAMIC TABLE DT_WINE_INVENTORY_STATUS
    TARGET_LAG = '15 minutes'
    WAREHOUSE = GIRODEFITO_WH
AS
SELECT 
    wine_id, wine_name, winery, wine_type, vintage_year,
    quantity_in_stock, reorder_level, purchase_price, selling_price,
    (selling_price - purchase_price) / NULLIF(selling_price, 0) * 100 AS margin_percentage,
    CASE 
        WHEN quantity_in_stock = 0 THEN '🔴 OUT OF STOCK'
        WHEN quantity_in_stock <= reorder_level / 2 THEN '🟠 CRITICAL'
        WHEN quantity_in_stock <= reorder_level THEN '🟡 REORDER'
        ELSE '🟢 ADEQUATE'
    END AS stock_status,
    CASE WHEN quantity_in_stock <= reorder_level THEN (reorder_level * 2) - quantity_in_stock ELSE 0 END AS suggested_reorder_qty
FROM WINE_INVENTORY;

SELECT '✅ Dynamic tables created!' AS STATUS;

In [ ]:
-- Query Dynamic Tables
SELECT * FROM DT_SALES_DASHBOARD ORDER BY order_date DESC LIMIT 10;
SELECT * FROM DT_CUSTOMER_LTV ORDER BY ltv_score DESC LIMIT 10;
SELECT * FROM DT_WINE_INVENTORY_STATUS WHERE stock_status != '🟢 ADEQUATE';

---

# 11. Demo Queries

🔍 **Sample analytical queries**

In [ ]:
-- ============================================================================
-- 📊 EXECUTIVE DASHBOARD
-- ============================================================================

SELECT 
    '🏢 Total Locations' AS metric, COUNT(DISTINCT location_id)::VARCHAR AS value
FROM LOCATIONS WHERE is_active = TRUE
UNION ALL
SELECT '💰 Total Revenue (90 days)', UDF_FORMAT_CURRENCY_IT(SUM(total_amount))
FROM ORDERS_FROM_STAGE WHERE order_timestamp >= DATEADD('day', -90, CURRENT_TIMESTAMP())
UNION ALL
SELECT '🧾 Total Orders (90 days)', COUNT(*)::VARCHAR
FROM ORDERS_FROM_STAGE WHERE order_timestamp >= DATEADD('day', -90, CURRENT_TIMESTAMP())
UNION ALL
SELECT '⚠️ Wine Inventory Alerts', COUNT(*)::VARCHAR
FROM DT_WINE_INVENTORY_STATUS WHERE stock_status != '🟢 ADEQUATE'
UNION ALL
SELECT '⭐ Premium Customers (Gold+)', COUNT(*)::VARCHAR
FROM CUSTOMERS WHERE membership_tier IN ('Gold', 'Platinum');

In [ ]:
-- ============================================================================
-- 🍝 MENU WITH WINE PAIRINGS
-- ============================================================================

SELECT 
    mc.category_name,
    mi.item_name,
    UDF_FORMAT_CURRENCY_IT(mi.price) AS price,
    CASE WHEN mi.is_vegan THEN '🌱' WHEN mi.is_vegetarian THEN '🥬' ELSE '' END AS dietary,
    REPEAT('🌶️', mi.spice_level) AS spice,
    UDF_WINE_PAIRING(mc.category_name, mi.spice_level > 2, 'Standard') AS wine_suggestion,
    UDF_SEASONAL_MENU_FLAG(mc.category_name, MONTH(CURRENT_DATE())) AS seasonal
FROM MENU_ITEMS mi
JOIN MENU_CATEGORIES mc ON mi.category_id = mc.category_id
WHERE mi.is_available = TRUE
ORDER BY mc.display_order, mi.price DESC;

In [ ]:
-- ============================================================================
-- 💧 SPRING WATER ANALYSIS
-- ============================================================================

SELECT 
    product_name,
    source_location,
    water_type,
    bottle_size_ml || 'ml' AS size,
    'pH ' || ph_level AS ph,
    mineral_content:calcium::NUMBER || 'mg Ca' AS calcium,
    mineral_content:magnesium::NUMBER || 'mg Mg' AS magnesium,
    UDF_FORMAT_CURRENCY_IT(selling_price) AS price,
    quantity_in_stock || ' units' AS stock
FROM SPRING_WATER_PRODUCTS
WHERE is_available = TRUE
ORDER BY bottle_size_ml DESC;

---

# 12. Cleanup

🧹 **Optional: Remove all demo objects**

> ⚠️ **Warning**: Uncomment and run only when you want to remove all demo objects.

In [ ]:
-- ============================================================================
-- 🧹 CLEANUP (OPTIONAL)
-- ============================================================================
-- ⚠️ WARNING: Uncomment to permanently delete all demo objects!
-- ============================================================================

/*
USE ROLE ACCOUNTADMIN;

-- Suspend tasks
ALTER TASK GIRODEFITO.PROD.TASK_PROCESS_ORDER_CHANGES SUSPEND;
ALTER TASK GIRODEFITO.PROD.TASK_CHECK_WINE_INVENTORY SUSPEND;

-- Drop database (cascades all objects)
DROP DATABASE IF EXISTS GIRODEFITO CASCADE;

-- Drop warehouse
DROP WAREHOUSE IF EXISTS GIRODEFITO_WH;

-- Drop roles
DROP ROLE IF EXISTS GIRODEFITO_ADMIN;
DROP ROLE IF EXISTS GIRODEFITO_USER;

SELECT '✅ Cleanup complete!' AS STATUS;
*/

---

# 📚 Summary

This demo showcased the following Snowflake features for **Il Giro de Fito**:

| Category | Features Demonstrated |
|----------|----------------------|
| **Tables** | Permanent, Transient, Temporary, Hybrid |
| **Views** | Standard, Secure, Materialized |
| **Open Format** | Apache Iceberg Tables (Internal) |
| **Data Loading** | Internal Stages, File Formats, COPY INTO |
| **Programmability** | Java & Python Stored Procedures, Python UDFs |
| **Data Pipelines** | Streams (CDC), Tasks (Scheduling) |
| **Real-time Analytics** | Dynamic Tables |
| **Security** | RBAC (Admin & User Roles) |

---

### 🍝 Buon appetito e buon lavoro con Snowflake!

*Il Giro de Fito - Tradizione Toscana dal 2018*